# Serial Communication and Robot Control for the myCobot 280 Labs
## Lab 7: Frames, Commands, State Feedback, and Safe Real-Robot Operation

Run this notebook from `~/venvs/mycobot`:

```bash
source ~/venvs/mycobot/bin/activate
jupyter lab
```

This is the first lab where you interact directly with the physical myCobot 280. You will inspect the serial protocol, send commands, read back robot state, compare commanded and measured motion, and use basic safety-stop procedures. The notebook connects low-level serial communication to the higher-level ROS 2 workflow.

**Important:** This notebook has two parts. In Part 1 (Tasks 1–15), you work with the serial interface directly — the ROS 2 controller must NOT be running. In Part 2 (Tasks 16–29), you use the ROS 2 controller — the serial object must be closed first.

In [ ]:
%matplotlib widget
import os
import time
import numpy as np
import matplotlib.pyplot as plt

The serial interface and ROS 2 imports are loaded in their respective sections below to keep the two parts cleanly separated.

## Part 1: Direct Serial Communication

In this part you communicate directly with the myCobot 280 through the serial interface. The ROS 2 controller (`mycobot_control`) must **not** be running — it would hold the serial port exclusively.

## Setup and Codebase Inspection

### 1. Inspect the codebase files

Before running any commands, read through the following files in the `Code` folder:
- `serial_iface.py` — the low-level serial wrapper
- `mycobot_control.py` — the ROS 2 controller node (used in Part 2)

Look for:
- How the serial port is opened and configured
- How frames are assembled and parsed
- How angles are encoded (int16, degrees × 100)
- How stop/resume commands are structured
- How the ROS 2 node bridges serial commands to topics

In [ ]:
# Confirm the serial interface file is available
import os
serial_iface_path = os.path.join(os.getcwd(), 'serial_iface.py')
print('serial_iface.py exists:', os.path.isfile(serial_iface_path))

### 2. Define the serial connection parameters

The myCobot 280 communicates over a UART serial connection. Define the expected parameters from the codebase.

In [ ]:
serial_port = ...   # Set the serial device used by the robot
baud_rate = ...     # Set the communication baud rate
timeout_s = ...     # Set an appropriate serial timeout

print(f'Port: {serial_port}, Baud: {baud_rate}, Timeout: {timeout_s} s')

## Communication Frames and Packet Structure

The myCobot 280 uses structured frames so the controller can identify command boundaries, interpret payloads, and detect invalid data. The frame format is:

```
FE FE LEN CMD [payload] FA
```

- `FE FE` — header (2 bytes)
- `LEN` — byte count from CMD through FA inclusive: `LEN = 1 + len(payload) + 1`
- `CMD` — command type byte
- `[payload]` — command-specific data (may be empty)
- `FA` — tail byte

### 3. Define the protocol constants

Define the header, tail, and key command bytes from the codebase.

In [ ]:
HEAD = 0xFE
TAIL = 0xFA

CMD_READ_ANGLES = 0x20
CMD_SEND_ANGLES = 0x22
CMD_RESUME = 0x28
CMD_STOP = 0x29

print(f'Header: 0x{HEAD:02X}, Tail: 0x{TAIL:02X}')
print(f'Read angles: 0x{CMD_READ_ANGLES:02X}')
print(f'Send angles: 0x{CMD_SEND_ANGLES:02X}')
print(f'Stop: 0x{CMD_STOP:02X}, Resume: 0x{CMD_RESUME:02X}')

### 4. Manually assemble a frame

Build the **stop** frame by hand. A stop command has no payload, so `LEN = 1 (CMD) + 1 (FA) = 2`. The full frame is: `FE FE 02 29 FA`.

Also build the **read-angles** request frame: `FE FE 02 20 FA`.

In [ ]:
# Stop frame
stop_frame = bytes([...])  # Build the protocol frame for the stop command
print('Stop frame:', stop_frame.hex(' '))

# Read-angles request frame
read_frame = bytes([...])  # Build the protocol frame for requesting joint angles
print('Read frame:', read_frame.hex(' '))

### 5. Write a frame-printing helper

Create a short function that prints any byte sequence as formatted hex for debugging. This is useful throughout the lab when inspecting raw serial traffic.

In [ ]:
def print_frame(label, frame_bytes):
    hex_str = ' '.join(f'{b:02X}' for b in frame_bytes)
    print(f'{label}: [{hex_str}] ({len(frame_bytes)} bytes)')

# Test with the stop frame
print_frame('Stop', stop_frame)
print_frame('Read', read_frame)

## Reading Robot State

### 6. Open the serial connection

Instantiate the `MyCobotSerialInterface` from the codebase wrapper. This opens the serial port — make sure the ROS 2 controller is NOT running.

**Note:** The serial interface uses an internal threading lock. Do not run multiple serial cells simultaneously.

In [ ]:
from serial_iface import MyCobotSerialInterface

ser = MyCobotSerialInterface(
    port=serial_port,
    baud=baud_rate,
    timeout=timeout_s,
    num_joints=6,
    stop_opcode=CMD_STOP,
    resume_opcode=CMD_RESUME,
)
print('Serial port opened.')

### 7. Read the current joint angles

Call `read_angles_deg()` to read the measured joint state. The returned values are in **degrees** (the hardware encoding divides int16 values by 100). These are measured encoder values, not commanded targets.

In [ ]:
measured_deg = ...  # Read the current joint angles from the robot

if measured_deg is not None:
    print('Measured joint angles (deg):', measured_deg)
else:
    print('No response from robot. Check connection and power.')

### 8. Convert and compare with a reference

Convert the measured angles to a NumPy array and compare with the expected home position (all zeros). Note: the myCobot 280 rarely reads exactly zero due to encoder resolution and mechanical play.

In [ ]:
if measured_deg is not None:
    measured_arr = np.array(measured_deg)
    home_deg = np.zeros(6)
    diff_deg = ...  # Compute the difference between the measured and home configurations

    print('Measured (deg):', measured_arr)
    print('Home (deg):    ', home_deg)
    print('Difference (deg):', diff_deg)

## Sending Position Commands

### 9. Define a safe position command

Define a small, safe joint-angle target in **degrees** and a speed level (1–100). Start with a very small motion from the current position.

**Caution:** Ensure the workspace is clear and the motion direction is understood before sending any command.

In [ ]:
# Small safe target — move joint 1 by ~10 degrees from current position
target_deg = ...  # Choose a small, safe six-joint target in degrees
speed = ...       # Choose a slow, safe command speed within the valid range

print('Target (deg):', target_deg)
print('Speed:', speed)

### 10. Send the position command

Send the target using the codebase wrapper. Start with small, well-understood motions and confirm the result before attempting larger movements.

In [ ]:
ser.send_angles_deg(target_deg, speed)
print('Position command sent.')

# Wait for the motion to complete
time.sleep(3.0)

### 11. Validate: compare commanded and measured angles

Read the robot state again after the motion and compute the difference between the commanded target and the measured position. Small residual differences are normal.

In [ ]:
measured_after = ser.read_angles_deg(timeout_s=0.5)

if measured_after is not None:
    measured_after_arr = np.array(measured_after)
    target_arr = np.array(target_deg)
    error_deg = ...  # Compute the command-to-measurement error

    print('Commanded (deg):', target_arr)
    print('Measured (deg): ', measured_after_arr)
    print('Error (deg):    ', error_deg)
    print('Max |error| (deg):', np.max(np.abs(error_deg)))

### 12. Plot commanded vs measured joint values

Visualise the commanded and measured values side by side for all six joints.

In [ ]:
if measured_after is not None:
    joints = np.arange(1, 7)
    fig, ax = plt.subplots(figsize=(8, 4))
    width = 0.3
    ax.bar(joints - width/2, target_arr, width, label='Commanded')
    ax.bar(joints + width/2, measured_after_arr, width, label='Measured')
    ax.set_xlabel('Joint')
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Commanded vs Measured Joint Angles')
    ax.legend()
    ax.grid(True, axis='y')
    plt.tight_layout()

## Stop and Resume Handling

### 13. Send a stop command

The stop command immediately halts robot motion. Use the codebase wrapper method — this is not optional but an essential part of safe robot operation.

In [ ]:
ser.stop_motion()
print('Stop command sent.')

### 14. Send a resume command

After a stop, the robot ignores motion commands until a resume is issued. Send the resume command and confirm that the robot accepts commands again.

In [ ]:
ser.resume_motion()
print('Resume command sent.')

# Verify by reading angles
check = ser.read_angles_deg(timeout_s=0.5)
if check is not None:
    print('Robot responding. Angles (deg):', check)
else:
    print('No response after resume.')

### 15. Close the serial connection

Before switching to the ROS 2 controller in Part 2, close the serial port. The controller needs exclusive access to the serial device.

**Reflection:** Why should stop functionality be treated as part of the normal control workflow, not as an exceptional feature? Consider what happens if you issue a command with an unintended target or if the robot moves in an unexpected direction.

In [ ]:
ser.close()
print('Serial port closed. You can now launch the ROS 2 controller.')

## Part 2: ROS 2 Controller Integration

In this part you use the ROS 2 controller node (`mycobot_control`) which handles serial communication internally. Launch the controller in a separate terminal before proceeding:

```bash
source ~/venvs/mycobot/bin/activate
ros2 launch mycobot_control mycobot_control.launch.py
```

**Unit convention change:** The direct serial interface uses **degrees** (hardware encoding). The ROS 2 topics use **radians** (ROS convention). The controller converts internally:
- `/mycobot/joint_command`: expects **radians** → controller does `math.degrees(x)`
- `/mycobot/joint_velocity`: expects **rad/s** → mapped to speed 0–100 internally
- `/mycobot/joint_states`: publishes positions in **radians**
- `/mycobot/ee_pose`: expects `[x_mm, y_mm, z_mm, roll_deg, pitch_deg, yaw_deg]`

## Position Mode and Velocity Mode

### 16. Position mode vs velocity mode

**Position mode** (`/mycobot/joint_command`): you specify a target configuration, and the controller drives the robot there. Appropriate when the desired end state is known.

**Velocity mode** (`/mycobot/joint_velocity`): you specify a motion rate, and the robot moves as long as the velocity command is active. Appropriate for continuous or incremental motion. Requires extra care:
- Deadband: velocities below 0.01 rad/s per joint are treated as zero
- Max speed: ~2.618 rad/s (150 deg/s)
- Zero velocity vector triggers an immediate stop
- Acceleration ramp: limited to 1.0 rad/s²

### 17. Inspect the ROS 2 topics

Use the ROS 2 CLI to verify the controller is running and inspect the available topics.

In [ ]:
# Run these in a terminal or uncomment to run from the notebook:
# !ros2 topic list
# !ros2 topic info /mycobot/joint_states
# !ros2 topic info /mycobot/joint_command
# !ros2 topic info /mycobot/joint_velocity
# !ros2 topic echo /mycobot/joint_states --once

## ROS 2 Node Integration

### 18. Create a notebook node

Create a small ROS 2 node that subscribes to `/mycobot/joint_states` and stores the most recent measured state.

In [ ]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64MultiArray


class SerialLabNode(Node):
    def __init__(self):
        super().__init__('serial_lab_node')
        self.last_js = None
        self.js_sub = self.create_subscription(
            JointState, '/mycobot/joint_states', self._js_cb, 10,
        )

    def _js_cb(self, msg):
        self.last_js = msg

### 19. Initialize and spin

Initialize `rclpy`, create the node, and spin once to receive the latest joint state.

In [ ]:
if not rclpy.ok():
    rclpy.init(args=None)

node = SerialLabNode()
rclpy.spin_once(node, timeout_sec=1.0)

if node.last_js is not None:
    print('Joint names:', node.last_js.name)
    print('Positions (rad):', list(node.last_js.position))
    print('Positions (deg):', [np.degrees(p) for p in node.last_js.position])
else:
    print('No joint state received. Is the controller running?')

### 20. Create a position-command publisher

Create a publisher for `/mycobot/joint_command`. This topic expects joint angles in **radians**.

In [ ]:
pos_pub = node.create_publisher(Float64MultiArray, '/mycobot/joint_command', 10)

### 21. Publish a safe position command (radians)

Send a small safe joint-position command in **radians** via the ROS 2 position-mode path. This is the higher-level equivalent of the direct `send_angles_deg()` used in Part 1.

In [ ]:
q_cmd_rad = ...  # Choose a small, safe six-joint position command in radians

msg = Float64MultiArray()
msg.data = ...  # Convert the position command to the message data format
pos_pub.publish(msg)
print(f'Published position command (rad): {q_cmd_rad}')

# Wait and read back
time.sleep(3.0)
rclpy.spin_once(node, timeout_sec=0.5)
if node.last_js is not None:
    print('Measured (rad):', list(node.last_js.position))

### 22. Create a velocity-command publisher

Create a publisher for `/mycobot/joint_velocity`. This topic expects joint velocities in **rad/s**.

In [ ]:
vel_pub = node.create_publisher(Float64MultiArray, '/mycobot/joint_velocity', 10)

### 23. Publish a safe velocity command

Send a very small velocity on one joint for a short duration, then send zeros to stop.

**Safety warning:**
- Keep velocities small (e.g., 0.1 rad/s on one joint)
- Keep duration short (e.g., 1–2 seconds)
- Be ready to send zeros or use the stop mechanism immediately
- Velocities below 0.01 rad/s are ignored (deadband)
- Sending `[0, 0, 0, 0, 0, 0]` triggers an immediate stop

In [ ]:
# Small safe velocity: rotate joint 1 slowly
qdot_cmd = ...  # Choose a small, safe six-joint velocity command
duration = ...  # Choose a short, safe motion duration

rate_hz = 20.0
msg = Float64MultiArray()

start = time.time()
while time.time() - start < duration:
    msg.data = ...  # Convert the velocity command to the message data format
    vel_pub.publish(msg)
    rclpy.spin_once(node, timeout_sec=0.0)
    time.sleep(1.0 / rate_hz)

# Stop: send zero velocity
msg.data = [0.0] * 6
vel_pub.publish(msg)
print('Velocity command completed. Stop sent.')

## Commanded vs Measured Motion

### 24. Record measured joint states over time

Send a position command and record the measured `/mycobot/joint_states` at regular intervals to observe the motion dynamics.

In [ ]:
# Send a target
q_target_rad = ...  # Choose a small, safe target configuration in radians
msg = Float64MultiArray()
msg.data = q_target_rad.tolist()
pos_pub.publish(msg)

# Record measured states
timestamps = []
measured_positions = []
start = time.time()
record_duration = 4.0

while time.time() - start < record_duration:
    rclpy.spin_once(node, timeout_sec=0.05)
    if node.last_js is not None:
        timestamps.append(time.time() - start)
        measured_positions.append(list(node.last_js.position))
    time.sleep(0.1)

timestamps = np.array(timestamps)
measured_positions = np.array(measured_positions)
print(f'Recorded {len(timestamps)} samples over {record_duration} s')

### 25. Plot the motion over time

Visualise how each joint moves toward the commanded target over time.

In [ ]:
if len(timestamps) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    for j in range(measured_positions.shape[1]):
        ax.plot(timestamps, np.degrees(measured_positions[:, j]), label=f'q{j+1}')
        ax.axhline(np.degrees(q_target_rad[j]), color=f'C{j}', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Measured joint motion toward commanded target')
    ax.legend(fontsize=7)
    ax.grid(True)
    plt.tight_layout()

### 26. Interpret commanded vs measured differences

Why might commanded and measured values differ? Consider:
- Motion still in progress (robot has not yet reached the target)
- Sampling delay (discrete read intervals vs continuous motion)
- Internal controller limits (acceleration ramp, speed cap)
- Mechanical play and encoder resolution

In [ ]:
# Space for student interpretation.

## Safety and Practical Precautions

### 27. Pre-motion safety checklist

Before enabling any robot motion, verify:
- Workspace is clear of obstacles and people
- Motion direction and magnitude are understood
- Stop mechanism is known and accessible
- Commands are modest during initial tests
- Serial port or controller is confirmed active

In [ ]:
# Students can use this cell to print a confirmation or checklist before running motion commands.
print('Safety checklist confirmed: [ ]')  # Students check this manually

### 28. Why real-robot operation requires extra caution

Briefly explain why direct communication with a real robot requires more caution than simulated motion. Consider physical consequences, irreversible actions, and the gap between model assumptions and hardware behaviour.

In [ ]:
# Space for student reflection.

## Reflection

### 29. Summary reflection

Summarise briefly:
1. What does the serial protocol contribute to reliable robot interaction?
2. Why does the frame structure (`FE FE LEN CMD ... FA`) matter?
3. Why is comparing commanded vs measured motion essential?
4. When is position mode appropriate vs velocity mode?

In [ ]:
# Space for student answers.

# Optional cleanup:
# node.destroy_node()
# rclpy.shutdown()